In [ ]:
import pandas as pd
import numpy as np
from sklift.datasets import fetch_x5


In [ ]:
##load in data
dataset = fetch_x5()

clients = dataset.data.clients
purchases = dataset.data.purchases
train = dataset.data.train

treatment = dataset.treatment
target = dataset.target

In [ ]:
print("Clients shape:", clients.shape)
print("Purchases shape:", purchases.shape)
print("Train shape:", train.shape)

Clients shape: (400162, 5)
Purchases shape: (45786568, 13)
Train shape: (200039, 1)


In [ ]:
##create a features table
features = train[["client_id"]].copy()

print(features.shape)
features.head()

(200039, 1)


In [ ]:
##check for invalid age
print(clients["age"].describe())

print("Age < 0:", (clients["age"] < 0).sum())
print("Age > 100:", (clients["age"] > 100).sum())

count    400162.000000
mean         46.488112
std          43.871218
min       -7491.000000
25%          34.000000
50%          45.000000
75%          59.000000
max        1901.000000
Name: age, dtype: float64
Age < 0: 96
Age > 100: 1049


In [ ]:
#set valid age and put age to feature
valid_age = clients.loc[
    (clients["age"] >= 0) & (clients["age"] <= 100),
    "age"
]

median_age = valid_age.median()

print("Median valid age:", median_age)

client_age = clients[["client_id", "age"]].copy()

client_age.loc[
    (client_age["age"] < 0) | (client_age["age"] > 100),
    "age"
] = median_age

features = features.merge(
    client_age,
    on="client_id",
    how="left"
)

features.head()

print(features["age"].describe())
print("Invalid ages remaining:",
      ((features["age"] < 0) | (features["age"] > 100)).sum())

Median valid age: 45.0
count    200039.000000
mean         46.359500
std          15.909756
min           0.000000
25%          34.000000
50%          45.000000
75%          59.000000
max         100.000000
Name: age, dtype: float64
Invalid ages remaining: 0


In [ ]:
##set gender

print(clients["gender"].value_counts(dropna=False))

gender_features = pd.get_dummies(
    clients[["client_id", "gender"]],
    columns=["gender"],
    prefix="gender",
    dtype=int
)

gender_features.head()

features = features.merge(
    gender_features,
    on="client_id",
    how="left"
)

features.head()

gender
U    185706
F    147649
M     66807
Name: count, dtype: int64


In [ ]:
# number of transactions
num_transactions = (
    purchases
    .groupby("client_id")["transaction_id"]
    .nunique()
    .rename("num_transactions")
    .reset_index()
)

num_transactions.head()

features = features.merge(
    num_transactions,
    on="client_id",
    how="left"
)

features["num_transactions"].describe()

In [ ]:
#number of uniq products purchased
num_unique_products = (
    purchases
    .groupby("client_id")["product_id"]
    .nunique()
    .rename("num_unique_products")
    .reset_index()
)

num_unique_products.head()
features = features.merge(
    num_unique_products,
    on="client_id",
    how="left"
)

features["num_unique_products"].describe()

In [ ]:
##number of stores visited
num_stores_visited = (
    purchases
    .groupby("client_id")["store_id"]
    .nunique()
    .rename("num_stores_visited")
    .reset_index()
)

num_stores_visited.head()
features = features.merge(
    num_stores_visited,
    on="client_id",
    how="left"
)

features["num_stores_visited"].describe()

In [ ]:
##total number of products customer purchased
total_quantity = (
    purchases
    .groupby("client_id")["product_quantity"]
    .sum()
    .rename("total_quantity")
    .reset_index()
)

features = features.merge(
    total_quantity,
    on="client_id",
    how="left"
)

features["total_quantity"].describe()



In [ ]:
##average item per transaction
features["avg_items_per_transaction"] = (
    features["total_quantity"] /
    features["num_transactions"]
)

features["avg_items_per_transaction"].describe()



In [ ]:
##total spending
total_spend = (
    purchases
    .groupby("client_id")["trn_sum_from_iss"]
    .sum()
    .rename("total_spend")
    .reset_index()
)

features = features.merge(
    total_spend,
    on="client_id",
    how="left"
)

features["total_spend"].describe()




In [ ]:
#average spending per transaction
features["avg_transaction_spend"] = (
    features["total_spend"] /
    features["num_transactions"]
)

features["avg_transaction_spend"].describe()




In [ ]:
##median spending for transactions
transaction_spend = (
    purchases
    .groupby(
        ["client_id", "transaction_id"],
        as_index=False
    )["trn_sum_from_iss"]
    .sum()
)

transaction_spend.head()

median_transaction_spend = (
    transaction_spend
    .groupby("client_id")["trn_sum_from_iss"]
    .median()
    .rename("median_transaction_spend")
    .reset_index()
)

features = features.merge(
    median_transaction_spend,
    on="client_id",
    how="left"
)

features["median_transaction_spend"].describe()

In [ ]:
## STD for spending
spend_std = (
    transaction_spend
    .groupby("client_id")["trn_sum_from_iss"]
    .std()
    .rename("spend_std")
    .reset_index()
)

features = features.merge(
    spend_std,
    on="client_id",
    how="left"
)

features["spend_std"].describe()




In [ ]:
##loyalty points bts
transaction_points = (
    purchases[
        [
            "client_id",
            "transaction_id",
            "regular_points_received",
            "regular_points_spent",
            "express_points_received",
            "express_points_spent"
        ]
    ]
    .drop_duplicates(
        subset=["client_id", "transaction_id"]
    )
)

transaction_points.head()


##total points earned for each customer

In [ ]:
##total points earned for each customer
regular_points_received = (
    transaction_points
    .groupby("client_id")["regular_points_received"]
    .sum()
    .rename("regular_points_received")
    .reset_index()
)

features = features.merge(
    regular_points_received,
    on="client_id",
    how="left"
)

features["regular_points_received"].describe()


##how much points spent per customer

In [ ]:
##how much points spent per customer
regular_points_spent = (
    transaction_points
    .groupby("client_id")["regular_points_spent"]
    .sum()
    .rename("regular_points_spent")
    .reset_index()
)

features = features.merge(
    regular_points_spent,
    on="client_id",
    how="left"
)

features["regular_points_spent"].describe()




In [ ]:
##total express points earned per customer
express_points_received = (
    transaction_points
    .groupby("client_id")["express_points_received"]
    .sum()
    .rename("express_points_received")
    .reset_index()
)

features = features.merge(
    express_points_received,
    on="client_id",
    how="left"
)

features["express_points_received"].describe()



##total express point spent per customer

In [ ]:
##total express point spent per customer
express_points_spent = (
    transaction_points
    .groupby("client_id")["express_points_spent"]
    .sum()
    .rename("express_points_spent")
    .reset_index()
)

features = features.merge(
    express_points_spent,
    on="client_id",
    how="left"
)

features["express_points_spent"].describe()




In [ ]:
##loyalty point net
features["net_regular_points_change"] = (
    features["regular_points_received"]
    + features["regular_points_spent"]
)

features["net_regular_points_change"].describe()




In [ ]:
##express point net
features["net_express_points_change"] = (
    features["express_points_received"]
    + features["express_points_spent"]
)

features["net_express_points_change"].describe()





In [ ]:

##days since last purchase
last_purchase = (
    purchases
    .groupby("client_id")["transaction_datetime"]
    .max()
    .rename("last_purchase_date")
    .reset_index()
)

last_purchase["last_purchase_date"] = pd.to_datetime(
    last_purchase["last_purchase_date"]
)

reference_date = last_purchase["last_purchase_date"].max()

print("Reference date:", reference_date)

last_purchase["days_since_last_purchase"] = (
    reference_date
    - last_purchase["last_purchase_date"]
).dt.days

days_since_last_purchase = last_purchase[
    ["client_id", "days_since_last_purchase"]
]

features = features.merge(
    days_since_last_purchase,
    on="client_id",
    how="left"
)

features["days_since_last_purchase"].describe()

Reference date: 2019-03-18 23:40:03


In [ ]:
##average day between transactions
transaction_dates = (
    purchases[
        ["client_id", "transaction_id", "transaction_datetime"]
    ]
    .drop_duplicates(
        subset=["client_id", "transaction_id"]
    )
)

transaction_dates["transaction_datetime"] = pd.to_datetime(
    transaction_dates["transaction_datetime"]
)

transaction_dates = transaction_dates.sort_values(
    ["client_id", "transaction_datetime"]
)

transaction_dates.head()

transaction_dates["days_since_previous"] = (
    transaction_dates
    .groupby("client_id")["transaction_datetime"]
    .diff()
    .dt.total_seconds()
    / 86400
)

avg_days_between_transactions = (
    transaction_dates
    .groupby("client_id")["days_since_previous"]
    .mean()
    .rename("avg_days_between_transactions")
    .reset_index()
)

features = features.merge(
    avg_days_between_transactions,
    on="client_id",
    how="left"
)

features["avg_days_between_transactions"].describe()

In [ ]:
##number of days btw first and last purchase
customer_activity_span = (
    transaction_dates
    .groupby("client_id")["transaction_datetime"]
    .agg(["min", "max"])
    .reset_index()
)

customer_activity_span["customer_activity_span"] = (
    customer_activity_span["max"]
    - customer_activity_span["min"]
).dt.days

customer_activity_span = customer_activity_span[
    ["client_id", "customer_activity_span"]
]

features = features.merge(
    customer_activity_span,
    on="client_id",
    how="left"
)

features["customer_activity_span"].describe()



##add treatment and target

In [ ]:
##add treatment and target
labels = pd.DataFrame({
    "client_id": train["client_id"].to_numpy(),
    "treatment": treatment.to_numpy(),
    "target": target.to_numpy()
})

labels.head()






In [ ]:
##merge
features = features.merge(
    labels,
    on="client_id",
    how="left",
    validate="one_to_one"
)

features.head()

In [ ]:
print(features.shape)
print(features.columns.tolist())





(200039, 25)
['client_id', 'age', 'gender_F', 'gender_M', 'gender_U', 'num_transactions', 'num_unique_products', 'num_stores_visited', 'total_quantity', 'avg_items_per_transaction', 'total_spend', 'avg_transaction_spend', 'median_transaction_spend', 'spend_std', 'regular_points_received', 'regular_points_spent', 'express_points_received', 'express_points_spent', 'net_regular_points_change', 'net_express_points_change', 'days_since_last_purchase', 'avg_days_between_transactions', 'customer_activity_span', 'treatment', 'target']


In [ ]:
#fill those with only 1 transaction  ( 0 for spend_std and avg between transaction)
features["spend_std"] = (
    features["spend_std"].fillna(0)
)

features["avg_days_between_transactions"] = (
    features["avg_days_between_transactions"].fillna(0)
)

In [ ]:

features.to_csv(
    "retailhero_causal_forest_features.csv",
    index=False
)



